In [1]:
import numpy as np, librosa, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [3]:
SR, N_FFT, HOP, N_MELS, MAX_LEN = 22050, 2048, 512, 128, 128
TARGET_SAMPLES = int(SR * 3.0)
THRESHOLD  = 0.374
MODEL_PATH = "checkpoint_9.keras"

MEL_MATRIX = tf.signal.linear_to_mel_weight_matrix(
    num_mel_bins=N_MELS, num_spectrogram_bins=N_FFT // 2 + 1,
    sample_rate=SR, lower_edge_hertz=0.0, upper_edge_hertz=SR / 2,
)

class PerSampleStandardization(layers.Layer):
    def call(self, x):
        mean = tf.reduce_mean(x, axis=[1, 2, 3], keepdims=True)
        std  = tf.math.reduce_std(x, axis=[1, 2, 3], keepdims=True)
        return (x - mean) / (std + 1e-6)

@tf.function(input_signature=[tf.TensorSpec(shape=[None], dtype=tf.float32)])
def tf_mel_spectrogram(waveform):
    stft = tf.signal.stft(waveform, frame_length=N_FFT, frame_step=HOP,
                          fft_length=N_FFT, window_fn=tf.signal.hann_window)
    mel = tf.matmul(tf.abs(stft) ** 2, MEL_MATRIX)
    mel_db = 10.0 * (tf.math.log(mel + 1e-6) / tf.math.log(10.0))
    mel_db = tf.maximum(mel_db - tf.reduce_max(mel_db), -80.0)
    mel_db = tf.transpose(mel_db)
    frames = tf.shape(mel_db)[1]
    mel_db = tf.cond(frames < MAX_LEN,
                     lambda: tf.pad(mel_db, [[0, 0], [0, MAX_LEN - frames]], constant_values=-80.0),
                     lambda: mel_db[:, :MAX_LEN])
    return tf.ensure_shape(mel_db, [N_MELS, MAX_LEN])

def preprocess(path):
    w = load_audio(path, sr=SR)
    if w.size > 0:
        t, _ = librosa.effects.trim(w, top_db=30)
        if t.size > 0: w = t
    if w.size == 0: w = np.zeros(TARGET_SAMPLES, dtype=np.float32)
    peak = np.max(np.abs(w))
    if peak > 0: w = w / peak
    w = w[:TARGET_SAMPLES] if w.size >= TARGET_SAMPLES else np.pad(w, (0, TARGET_SAMPLES - w.size))
    mel = tf_mel_spectrogram(tf.convert_to_tensor(w.astype(np.float32)))
    return mel.numpy()[np.newaxis, ..., np.newaxis]     # (1, N_MELS, MAX_LEN, 1)

FFMPEG para usar cualqueir tipo de audio para pasarlo a wav

In [4]:
import subprocess, numpy as np

def load_audio(path, sr=SR):
    # hacer el decode de cualquier formato a mono float32 @ sr
    cmd = ["ffmpeg", "-v", "error", "-i", path,
           "-ac", "1", "-ar", str(sr), "-f", "f32le", "-"]
    raw = subprocess.run(cmd, capture_output=True, check=True).stdout
    return np.frombuffer(raw, dtype=np.float32).copy()

In [9]:

model = keras.models.load_model(MODEL_PATH,custom_objects={"PerSampleStandardization": PerSampleStandardization})

def predict(path):
    prob_real = float(model.predict(preprocess(path), verbose=0).ravel()[0])
    prob_fake = 1.0 - prob_real
    if prob_real >= THRESHOLD:
        label, margin = "real", (prob_real - THRESHOLD) / (1.0 - THRESHOLD)
    else:
        label, margin = "fake", (THRESHOLD - prob_real) / THRESHOLD
    return {"label": label, "p_real": prob_real, "p_fake": prob_fake, "confidence": margin}

r = predict("/content/diego_real.mp4")
band = "baja" if r["confidence"] < 0.33 else "media" if r["confidence"] < 0.66 else "alta"
print(f"{r['label'].upper()}  |  P(real)={r['p_real']:.3f}  P(fake)={r['p_fake']:.3f}")
print(f"Confianza en la prediccion: {r['confidence']:.2f} ({band})")

REAL  |  P(real)=0.728  P(fake)=0.272
Confianza en la prediccion: 0.57 (medium)
